# Financial Tweet Sentiment Classification — Final Pipeline
## Group 42 — Text Mining 2025/2026, NOVA IMS

Single linear flow:
load data → preprocess → fully fine-tune **Twitter-RoBERTa** → save its test
softmax probabilities → fully fine-tune **FinBERT** → save its test softmax
probabilities → average the two probability tensors → argmax → write
`pred_42.csv`.

**Runtime**: ~16 minutes per model (~32 minutes total) on Apple-Silicon MPS.
About 4× that on CPU.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd().parent))
from src import DATA_DIR, MODELS_DIR, OUTPUTS_DIR, RANDOM_STATE
from src.preprocessing import pp_transformer

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print("Environment ready.")


## 1. Load and preprocess data


In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print(f"Training: {len(train_df):,} labelled tweets")
print(f"Test:     {len(test_df):,} unlabelled tweets")

print("\nApplying pp_transformer preprocessing...")
X_train = np.array([pp_transformer(t) for t in train_df["text"].values])
X_test = np.array([pp_transformer(t) for t in test_df["text"].values])
y_train = train_df["label"].values.astype(int)
test_ids = test_df["id"].values
print(f"Done. Train={len(X_train):,}  Test={len(X_test):,}")


## 2. Define the fine-tune-and-predict helper
Fully fine-tunes a single encoder for three epochs on the full training set
and returns the softmax probabilities on the test set. We use this twice
below: once for Twitter-RoBERTa and once for FinBERT.


In [ ]:
def softmax_np(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def full_finetune_and_predict(
    checkpoint, model_tag,
    X_train, y_train, X_test,
    max_length=128, batch_size=16, epochs=3,
    learning_rate=1e-5, weight_decay=0.01, warmup_ratio=0.1,
):
    print(f"\n=== Full fine-tune: {model_tag} ({checkpoint}) ===")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=3, ignore_mismatched_sizes=True,
    )
    model.config.id2label = {0: "Bearish", 1: "Bullish", 2: "Neutral"}
    model.config.label2id = {v: k for k, v in model.config.id2label.items()}

    def tok(batch):
        return tokenizer(batch["text"], padding=False, truncation=True,
                         max_length=max_length)

    train_ds = Dataset.from_dict({"text": list(X_train), "label": list(y_train)})
    train_ds = train_ds.map(tok, batched=True, remove_columns=["text"])
    # Trainer requires an eval set even with eval_strategy='no' to satisfy
    # internal init paths.
    eval_ds = Dataset.from_dict({"text": list(X_train[:200]), "label": list(y_train[:200])})
    eval_ds = eval_ds.map(tok, batched=True, remove_columns=["text"])

    args = TrainingArguments(
        output_dir=str(MODELS_DIR / f"final_{model_tag}"),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=learning_rate,
        warmup_ratio=warmup_ratio,
        weight_decay=weight_decay,
        eval_strategy="no",
        save_strategy="no",
        logging_steps=100,
        report_to=[],
        seed=RANDOM_STATE,
        fp16=False,
        bf16=False,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
    )

    trainer.train()

    print(f"\nPredicting test softmax probabilities for {model_tag}...")
    test_ds = Dataset.from_dict({"text": list(X_test)})
    test_ds = test_ds.map(tok, batched=True, remove_columns=["text"])
    out = trainer.predict(test_ds)
    probs = softmax_np(out.predictions, axis=-1)
    print(f"  shape={probs.shape}")
    return probs


## 3. Component 1 — Twitter-RoBERTa


In [ ]:
probs_roberta = full_finetune_and_predict(
    checkpoint="cardiffnlp/twitter-roberta-base-sentiment-latest",
    model_tag="roberta",
    X_train=X_train, y_train=y_train, X_test=X_test,
)
print(f"Twitter-RoBERTa argmax distribution:")
print(pd.Series(probs_roberta.argmax(-1)).value_counts().sort_index().to_dict())


## 4. Component 2 — FinBERT


In [ ]:
probs_finbert = full_finetune_and_predict(
    checkpoint="ProsusAI/finbert",
    model_tag="finbert",
    X_train=X_train, y_train=y_train, X_test=X_test,
)
print(f"FinBERT argmax distribution:")
print(pd.Series(probs_finbert.argmax(-1)).value_counts().sort_index().to_dict())


## 5. Ensemble: average probabilities, argmax


In [ ]:
probs_ensemble = (probs_roberta + probs_finbert) / 2
preds_ensemble = probs_ensemble.argmax(-1).astype(int)

print(f"Ensemble distribution:")
print(pd.Series(preds_ensemble).value_counts().sort_index().to_dict())
disagree = (probs_roberta.argmax(-1) != probs_finbert.argmax(-1)).sum()
print(f"\nModels disagree on {disagree} of {len(preds_ensemble)} test tweets")


## 6. Write `pred_42.csv`


In [ ]:
out = pd.DataFrame({"id": test_ids, "label": preds_ensemble})
out_path = OUTPUTS_DIR / "pred_42.csv"
out.to_csv(out_path, index=False)
print(f"Saved {len(out):,} predictions to {out_path}")

# Sanity check
assert list(out.columns) == ["id", "label"]
assert out["label"].isin([0, 1, 2]).all()
assert out.isna().sum().sum() == 0
print("OK —", out["label"].value_counts().sort_index().to_dict())
